In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')

Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v6_example
Connected to future database: 10


In [3]:
import pandas as pd

print("================================================================================")
print(" 🔍 AUTO-SCAN LAPORAN PERUBAHAN DATA: APRIL (DB_OLD) vs JUNI (DB_NEW) 🔍 ")
print("================================================================================")

# 1. Ambil daftar SEMUA tabel secara otomatis dari DB_NEW (Juni)
query_get_tables = f"SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = '{config['db_new']['database']}'"
cursor_new.execute(query_get_tables)
semua_tabel = [row['TABLE_NAME'] for row in cursor_new.fetchall()]

print(f"Total ada {len(semua_tabel)} tabel yang sedang dipindai...\n")

tabel_berubah = 0

# 2. Looping untuk menghitung selisih data satu per satu
for tabel in semua_tabel:
    try:
        # Hitung jumlah baris di DB_OLD (April)
        cursor_old.execute(f"SELECT COUNT(*) as total FROM `{tabel}`")
        hasil_old = cursor_old.fetchone()
        jml_april = hasil_old['total'] if hasil_old else 0
        
        # Hitung jumlah baris di DB_NEW (Juni)
        cursor_new.execute(f"SELECT COUNT(*) as total FROM `{tabel}`")
        hasil_new = cursor_new.fetchone()
        jml_juni = hasil_new['total'] if hasil_new else 0
        
        selisih = jml_juni - jml_april
        
        # 3. Tampilkan hanya yang berubah
        if selisih > 0:
            print(f"📈 {tabel.ljust(30)}: NAMBAH {selisih} baris (April: {jml_april} -> Juni: {jml_juni})")
            tabel_berubah += 1
        elif selisih < 0:
            print(f"📉 {tabel.ljust(30)}: BERKURANG {abs(selisih)} baris (April: {jml_april} -> Juni: {jml_juni})")
            tabel_berubah += 1
            
    except Exception as e:
        # Mengabaikan error jika ternyata tabel tersebut BARU dibuat di Juni dan belum ada di April
        print(f"🆕 {tabel.ljust(30)}: Tabel baru (Tidak ada di DB_OLD)")
        tabel_berubah += 1

print("\n================================================================================")
if tabel_berubah == 0:
    print("✅ AMAN! Tidak ada perubahan jumlah data sama sekali di semua tabel.")
else:
    print(f"⚠️ Ditemukan perbedaan jumlah data pada {tabel_berubah} tabel.")
print("================================================================================")

 🔍 AUTO-SCAN LAPORAN PERUBAHAN DATA: APRIL (DB_OLD) vs JUNI (DB_NEW) 🔍 
Total ada 115 tabel yang sedang dipindai...

📈 absensi                       : NAMBAH 941 baris (April: 13444 -> Juni: 14385)
🆕 cache                         : Tabel baru (Tidak ada di DB_OLD)
🆕 cache_locks                   : Tabel baru (Tidak ada di DB_OLD)
📈 catatan_kelas                 : NAMBAH 936 baris (April: 12797 -> Juni: 13733)
📈 catatan_kelas_tag             : NAMBAH 68 baris (April: 999 -> Juni: 1067)
📈 catatan_siswa                 : NAMBAH 33 baris (April: 1502 -> Juni: 1535)
🆕 failed_jobs                   : Tabel baru (Tidak ada di DB_OLD)
📈 file_rapor_siswa              : NAMBAH 26 baris (April: 1506 -> Juni: 1532)
📈 form_calon                    : NAMBAH 36 baris (April: 184 -> Juni: 220)
📈 histori_pengajuan             : NAMBAH 1 baris (April: 79 -> Juni: 80)
📈 history_rapor                 : NAMBAH 36 baris (April: 1366 -> Juni: 1402)
📈 jadwal                        : NAMBAH 7 baris (April: 551